# Chapter 4 — Attention

Attention is one of the most important ideas behind modern Large Language Models.

In this chapter, we will understand how a model decides which parts of the input are important when processing a particular token.

We will learn:

- Why attention is needed
- Query, Key, and Value
- Attention scores
- Softmax
- Attention weights
- Weighted aggregation
- Scaled attention
- Attention from scratch using NumPy
- Attention using PyTorch
- Experiments

By the end of this chapter, we should be able to explain and implement:

Attention(Q, K, V)
= softmax(QKᵀ / √dₖ)V

## 1. Why Do We Need Attention?

Consider the sentence:

"The animal didn't cross the road because it was too tired."

To understand the sentence, the model needs to determine what "it" refers to.

Here, "it" refers to "animal".

The two words are separated by several other words.

A language model therefore needs a mechanism that can look at different parts of the context and determine which tokens are important.

This is the basic idea behind attention.

Instead of only asking:

"What token came before me?"

attention asks:

"Which tokens in the context are important for me?"

## 2. The Basic Idea

Suppose we have:

"I love machine learning"

When processing the word "learning", some words may be more relevant than others.

Conceptually:

learning
   ↓
looks at all previous/context tokens
   ↓
I       → low importance
love    → medium importance
machine → high importance

Attention assigns different weights to different pieces of information.

The model can then combine the important information to create a better representation.

## 3. Query, Key, and Value

Attention uses three representations:

### Query

The Query represents what the current token is looking for.

Think:

"What information do I need?"

### Key

The Key represents what a token can be matched against.

Think:

"What kind of information do I contain?"

### Value

The Value contains the actual information that will be used.

Think:

"What information should I provide if I am relevant?"

A simple analogy:

Query → What am I looking for?

Key → What do I match against?

Value → What information do I receive?

## 4. Attention in Three Steps

Attention can be understood as three main operations.

### Step 1 — Calculate similarity

Compare the Query with every Key.

### Step 2 — Convert similarity into weights

Use softmax to convert the scores into probabilities.

### Step 3 — Combine the Values

Use the attention weights to calculate a weighted sum of the Values.

The process is:

Query + Keys
     ↓
Similarity Scores
     ↓
Softmax
     ↓
Attention Weights
     ↓
Weighted Values
     ↓
Output

## 5. Attention Equation

The basic attention equation is:

Attention(Q, K, V)
= softmax(QKᵀ)V

There are three important steps hidden inside this equation.

First:

Scores = QKᵀ

Then:

Weights = softmax(Scores)

Finally:

Output = Weights × V

Later, we will add scaling:

Attention(Q, K, V)
= softmax(QKᵀ / √dₖ)V

## 6. Dot Product Similarity

We need a way to measure how similar a Query is to a Key.

A simple method is the dot product.

For example:

Q = [1, 0]

K₁ = [1, 0]

K₂ = [0, 1]

Calculate:

Q · K₁
= 1×1 + 0×0
= 1

Q · K₂
= 1×0 + 0×1
= 0

Therefore:

Q is more similar to K₁ than K₂.

In [2]:
import numpy as np

Q=np.array([1.0,0.0])

K1=np.array([1.0,0.0])
K2=np.array([0.0,1.0])

score_1=Q@K1
score_2=Q@K2

print("Score with K1:",score_1)
print("Score with K2:",score_2)

Score with K1: 1.0
Score with K2: 0.0


### What happened?

The `@` operator performs matrix multiplication / dot product.

For K₁:

Q = [1, 0]
K₁ = [1, 0]

Therefore:

1×1 + 0×0 = 1

For K₂:

Q = [1, 0]
K₂ = [0, 1]

Therefore:

1×0 + 0×1 = 0

So the Query has a stronger relationship with K₁.

These values are called attention scores.

## 7. Multiple Keys and Values

In a real model, we don't have only two Keys.

Suppose we have three:

K₁
K₂
K₃

And each Key has a corresponding Value:

V₁
V₂
V₃

The Query is compared against every Key.

For example:

Q → K₁ → score
Q → K₂ → score
Q → K₃ → score

This gives us a set of attention scores.

In [4]:
Q=np.array([1.0,0.0])

K=np.array([
    [1.0,0.0],
    [0.0,1.0],
    [1.0,1.0]
])

scores=K@Q
print("Attention scores:")
print(scores)

Attention scores:
[1. 0. 1.]


The scores are:

[1, 0, 1]

This tells us that the Query has:

- high similarity with K₁
- low similarity with K₂
- high similarity with K₃

But these are raw scores.

We want to turn them into weights that tell us how much attention each Key receives.

For that, we use softmax.

## 8. Softmax

Softmax converts a collection of numbers into a probability distribution.

The formula is:

softmax(xᵢ) = exp(xᵢ) / Σ exp(xⱼ)

The resulting values:

- are between 0 and 1
- sum to 1

For example:

[1, 0]

becomes approximately:

[0.731, 0.269]

So the first item receives more attention.

In [5]:
def softmax(x):
  exp_x=np.exp(x)
  return exp_x/np.sum(exp_x)

In [6]:
scores=np.array([1.0,0.0])
weights=softmax(scores)

print("Scores:",scores)
print("Attention weights:",weights)
print("Sum:",weights.sum())

Scores: [1. 0.]
Attention weights: [0.73105858 0.26894142]
Sum: 1.0


### Understanding the weights

We started with:

Scores:

[1, 0]

After softmax:

[0.731, 0.269]

We can interpret this as:

K₁ → 73.1% attention
K₂ → 26.9% attention

The higher-scoring Key receives a higher attention weight.

The weights always sum to 1.

## 9. Values

Now we introduce Values.

Suppose:

V₁ = [10, 20]

V₂ = [30, 40]

And our attention weights are:

[0.731, 0.269]

The final output is a weighted combination of the Values:

Output
= 0.731 × V₁
+ 0.269 × V₂

Therefore, the Value associated with the higher attention weight contributes more strongly to the output.

In [7]:
V=np.array([
    [10.0,20.0],
    [30.0,40.0]
])

output=weights @ V

print("Attention output:")
print(output)

Attention output:
[15.37882843 25.37882843]


### What did we just do?

We performed the complete attention process:

1. Compare Query with Keys.
2. Get similarity scores.
3. Apply softmax.
4. Get attention weights.
5. Use the weights to combine Values.

In short:

Q + K
 ↓
Scores
 ↓
Softmax
 ↓
Weights
 ↓
Weighted V
 ↓
Output

## 10. Attention From Scratch

We can now combine everything into one function.

The basic attention equation is:

Attention(Q, K, V)
= softmax(QKᵀ)V

The three operations are:

Scores = QKᵀ

Weights = softmax(Scores)

Output = Weights × V

In [8]:
def attention(Q,K,V):
  scores=Q@ K.T

  weights=softmax(scores)

  output=weights @ V

  return output,weights

Let's test our attention function with one Query and two Keys.

Our Query is:

Q = [1, 0]

Our Keys are:

K₁ = [1, 0]
K₂ = [0, 1]

And our Values are:

V₁ = [10, 20]
V₂ = [30, 40]

In [9]:
Q = np.array([[1.0, 0.0]])

K = np.array([
    [1.0, 0.0],
    [0.0, 1.0]
])

V = np.array([
    [10.0, 20.0],
    [30.0, 40.0]
])

output, weights = attention(Q, K, V)

print("Attention weights:")
print(weights)

print("\nAttention output:")
print(output)

Attention weights:
[[0.73105858 0.26894142]]

Attention output:
[[15.37882843 25.37882843]]


In [10]:
Q = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

K = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

V = np.array([
    [10.0, 20.0],
    [30.0, 40.0],
    [50.0, 60.0]
])

scores = Q @ K.T

print("Attention scores:")
print(scores)

Attention scores:
[[1. 0. 1.]
 [0. 1. 1.]
 [1. 1. 2.]]


In [11]:
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
print("Scores shape:", scores.shape)

Q shape: (3, 2)
K shape: (3, 2)
V shape: (3, 2)
Scores shape: (3, 3)


In [12]:
def softmax_rows(x):

    # Subtract maximum for numerical stability
    x = x - np.max(x, axis=1, keepdims=True)

    exp_x = np.exp(x)

    return exp_x / np.sum(
        exp_x,
        axis=1,
        keepdims=True
    )

In [13]:
weights = softmax_rows(scores)

print("Attention weights:")
print(weights)

print("\nRow sums:")
print(weights.sum(axis=1))

Attention weights:
[[0.4223188  0.1553624  0.4223188 ]
 [0.1553624  0.4223188  0.4223188 ]
 [0.21194156 0.21194156 0.57611688]]

Row sums:
[1. 1. 1.]


In [14]:
output = weights @ V

print("Attention output:")
print(output)

Attention output:
[[30.         40.        ]
 [35.3391279  45.3391279 ]
 [37.28350654 47.28350654]]


In [15]:
def scaled_attention(Q, K, V):

    d_k = K.shape[-1]

    scores = (Q @ K.T) / np.sqrt(d_k)

    weights = softmax_rows(scores)

    output = weights @ V

    return output, weights, scores

In [16]:
output, weights, scores = scaled_attention(Q, K, V)

print("Scaled scores:")
print(scores)

print("\nAttention weights:")
print(weights)

print("\nOutput:")
print(output)

Scaled scores:
[[0.70710678 0.         0.70710678]
 [0.         0.70710678 0.70710678]
 [0.70710678 0.70710678 1.41421356]]

Attention weights:
[[0.40111209 0.19777581 0.40111209]
 [0.19777581 0.40111209 0.40111209]
 [0.24825508 0.24825508 0.50348984]]

Output:
[[30.         40.        ]
 [34.06672556 44.06672556]
 [35.1046953  45.1046953 ]]
